In [0]:
import re
import requests

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    ArrayType,
)

from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    to_date,
    explode,
    split,
    trim,
    lower,
    expr,
    avg,
    count as spark_count,
    sum as spark_sum,
    max as spark_max,
)

from delta.tables import DeltaTable

In [0]:
bronze_schema = "bronze"
silver_schema = "silver"
gold_schema = "gold"

In [0]:
dataset2_file_path = "/Volumes/bosch/bronze/dataset2/alt_fuel_stations (Mar 2 2026).csv"
dataset3_file_path_emissions = "/Volumes/bosch/bronze/dataset3/emissions.csv"
dataset3_file_path_vehicles = "/Volumes/bosch/bronze/dataset3/vehicles.csv"

In [0]:
# Get the maximum model_year from the bronze.model_years table, excluding the placeholder value 9999
max_year = (
    spark.table(f"{bronze_schema}.model_years")
         .filter("model_year <> 9999")
         .select(spark_max("model_year").alias("max_year"))
         .collect()[0]["max_year"]
)


In [0]:
# 1) Get all unique make/model pairs for the latest model year
driver_df = (
    spark.table(f"{bronze_schema}.vehicle_models")
         .filter(col("model_year") == lit(int(max_year)))
         .select("make", "model")
         .distinct()
)  


In [0]:
def clean_col(c: str) -> str:
    # Remove leading/trailing spaces and convert to lowercase
    c = c.strip().lower()
    # Replace forbidden characters with underscore
    c = re.sub(r"[ ,;{}()\n\t=]+", "_", c)
    # Remove any remaining non-alphanumeric or underscore characters
    c = re.sub(r"[^a-z0-9_]", "", c)
    # Normalize consecutive underscores and remove leading/trailing underscores
    c = re.sub(r"_+", "_", c).strip("_")
    return c


In [0]:
def union_all(dfs):
    # Return None if the input list is empty
    if not dfs:
        return None
    out = dfs[0]
    # Union all DataFrames by column name, allowing missing columns to be filled with nulls
    for d in dfs[1:]:
        out = out.unionByName(d, allowMissingColumns=True)
    return out
